# Foundry to SQL Managed Instance over a private path — end to end

Runs from **nothing** to a working agent, and is safe to re-run from **any partially-deployed
state**. Every cell checks before it creates: existing resources are reused and reported, never
duplicated.

If your infrastructure already exists, Phases 1 to 3 detect that and skip — run the notebook top to
bottom regardless.

### End state

```
Foundry agent  (public)
      |  Entra token, audience api://<mcp-app-client-id>
      v
SQL MCP Server  (Data API builder on Container Apps, VNet-injected)
      |  managed identity, TCP 1433, VNet-local FQDN
      v
Azure SQL Managed Instance  --  mcp schema, curated views, mcp_reader
```

Foundry stays public and authenticates with a token. Network isolation protects the path from MCP
to the data, which is where the data actually sits.

### Phases

| | Phase | Creates | Skips if present |
|---|---|---|---|
| 1 | Foundation | Resource group, monitoring, identity, VNet, subnets | yes |
| 2 | SQL Managed Instance | The instance — **4 to 6 hours** | yes |
| 3 | Foundry and Entra | Account, project, model, MCP app registration | yes |
| 4 | Data contract | `mcp` schema, curated views, grants, DAB entities | idempotent |
| 5 | Registry | Premium ACR, image, private endpoint, lockdown | yes |
| 6 | Container Apps | VNet-injected environment, container app | yes |
| 7 | Connection and agent | Foundry MCP connection, prompt agent | yes |
| 8 | Close and validate | Disable public SQL, prove the denials | — |

### Before you start

- **VS Code** with the *Polyglot Notebooks* extension and the **PowerShell** kernel selected.
  This will not run in Azure Cloud Shell.
- **Azure CLI**, signed in.
- **sqlcmd** — `winget install Microsoft.Sqlcmd` (needed in Phase 4).
- Permission to create the resources above, plus Entra application registration.

Reasoning for each step is in [`docs/demo-portal-runbook.md`](../docs/demo-portal-runbook.md);
choosing what to expose is in
[`docs/configure-for-your-database.md`](../docs/configure-for-your-database.md).

---
## Phase 0 — Configure

### 0.1 — Set your values

**The only cell you need to edit.**

Leave a name as-is to have it created; point it at an existing resource to reuse that instead.
`$rg` and `$vnetRg` are frequently different — the VNet hosting SQL MI often lives in a network
resource group. Setting both the same when they differ is the most common cause of
`ResourceNotFound` later.

In [ ]:
# -- Azure context ----------------------------------------------------------
$subscriptionId = '<subscription-id>'
$tenantId       = '<tenant-id>'
$location       = 'eastus2'
$rg             = 'rg-sql-mcp'

# -- Network ----------------------------------------------------------------
$vnetRg         = $rg                      # change if the VNet lives elsewhere
$vnetName       = 'vnet-sql-mcp'
$vnetPrefix     = '10.60.0.0/16'           # only used when creating a new VNet
$acaSubnet      = 'snet-aca-mcp'
$acaPrefix      = '10.60.1.0/27'           # /27 minimum for workload profiles
$peSubnet       = 'snet-private-endpoints'
$pePrefix       = '10.60.2.0/28'
$miSubnet       = 'snet-sqlmi'             # only created when SQL MI is created
$miPrefix       = '10.60.3.0/27'

# -- SQL Managed Instance ---------------------------------------------------
$miName         = 'sqlmi-sql-mcp'
$databaseName   = 'AgentData'
$miVCores       = 4
$miStorageGB    = 32
$miEntraAdmin   = ''                       # blank = the signed-in user

# -- Monitoring and identity ------------------------------------------------
$lawName        = 'log-sql-mcp'
$appInsightsName= 'appi-sql-mcp'
$identityName   = 'id-mcp-sql-mcp'

# -- Foundry ----------------------------------------------------------------
$foundryAccount = 'aif-sql-mcp'
$foundryProject = 'proj-sql-mcp'
$modelName      = 'gpt-5.4-mini'
$modelVersion   = '2026-03-17'
$modelDeployment= 'gpt-5.4-mini'
$modelCapacity  = 10

# -- Entra application for MCP ----------------------------------------------
$mcpAppName     = 'sql-mcp-api'

# -- Registry and runtime ---------------------------------------------------
$acrName        = 'acrsqlmcpsuffix'        # lowercase alphanumeric, globally unique
$envName        = 'cae-sql-mcp'
$appName        = 'app-sql-mcp'

Write-Host 'Values set.' -ForegroundColor Green

### 0.2 — Sign in and load helpers

Defines the helper functions every later cell uses. Run this once per session — if the kernel
restarts, run it again before continuing.

In [ ]:
$ErrorActionPreference = 'Stop'

# az login --tenant $tenantId          # uncomment if not already signed in
az account set --subscription $subscriptionId 2>$null

$acct = az account show -o json | ConvertFrom-Json
Write-Host ("Subscription : {0}" -f $acct.name)
Write-Host ("Signed in as : {0}" -f $acct.user.name)
if ($acct.tenantId -ne $tenantId) { Write-Warning 'Signed-in tenant does not match the configured tenant.' }

# -- Helpers ----------------------------------------------------------------
function Write-Created($what) { Write-Host ("  CREATED  {0}" -f $what) -ForegroundColor Green }
function Write-Reused($what)  { Write-Host ("  REUSED   {0}" -f $what) -ForegroundColor DarkGray }
function Write-Failed($what)  { Write-Host ("  FAILED   {0}" -f $what) -ForegroundColor Red }
function Write-Note($what)    { Write-Host ("  NOTE     {0}" -f $what) -ForegroundColor Yellow }

function Get-RepoRoot {
    $p = "$PWD"
    while ($p -and -not (Test-Path (Join-Path $p 'azure.yaml'))) {
        $parent = Split-Path -Parent $p
        if ($parent -eq $p) { return $null }
        $p = $parent
    }
    return $p
}

# Returns the parsed object when the resource exists, otherwise $null.
function Get-AzJson {
    param([Parameter(Mandatory)][string[]]$CliArgs)
    $out = & az @CliArgs -o json 2>$null
    if ($LASTEXITCODE -ne 0 -or [string]::IsNullOrWhiteSpace($out)) { return $null }
    try { return $out | ConvertFrom-Json } catch { return $null }
}

function Get-SqlMi {
    $m = Get-AzJson @('sql','mi','show','-g',$rg,'-n',$miName)
    if (-not $m) { $m = (Get-AzJson @('sql','mi','list')) | Where-Object { $_.name -eq $miName } | Select-Object -First 1 }
    return $m
}

$repoRoot = Get-RepoRoot
if (-not $repoRoot) { Write-Warning 'Repository root not found. Set $repoRoot manually before Phase 4.' }
else { Write-Host ("Repo root    : {0}" -f $repoRoot) }
Write-Host 'Helpers loaded.' -ForegroundColor Green

### 0.3 — Current state

**Creates nothing.** Run at any point to see exactly where the deployment stands — before you
start, after a failure, or to confirm the end state. This is what makes resuming from a
half-deployed state safe.

In [ ]:
$state = [System.Collections.Generic.List[object]]::new()
function S($phase, $resource, $exists, $detail) {
    $state.Add([pscustomobject]@{
        Phase = $phase; Resource = $resource
        Status = $(if ($exists) { 'EXISTS' } else { 'missing' })
        Detail = $detail })
}

S 1 'Resource group'      ((az group exists --name $rg) -eq 'true') $rg

$lawObj = Get-AzJson @('monitor','log-analytics','workspace','show','-g',$rg,'-n',$lawName)
S 1 'Log Analytics'       ($null -ne $lawObj) $lawName

$aiObj = Get-AzJson @('monitor','app-insights','component','show','-g',$rg,'-a',$appInsightsName)
S 1 'App Insights'        ($null -ne $aiObj) $appInsightsName

$uamiObj = Get-AzJson @('identity','show','-g',$rg,'-n',$identityName)
S 1 'Managed identity'    ($null -ne $uamiObj) $identityName

$vnetObj = Get-AzJson @('network','vnet','show','-g',$vnetRg,'-n',$vnetName)
S 1 'Virtual network'     ($null -ne $vnetObj) $vnetName
if ($vnetObj) {
    foreach ($sn in @($acaSubnet, $peSubnet)) {
        $found = $vnetObj.subnets | Where-Object { $_.name -eq $sn }
        S 1 ("  subnet " + $sn) ($null -ne $found) $(if ($found) { $found.addressPrefix } else { '-' })
    }
}

$miObj = Get-SqlMi
S 2 'SQL Managed Instance' ($null -ne $miObj) $(if ($miObj) { "$($miObj.state) - $($miObj.fullyQualifiedDomainName)" } else { $miName })

$fdyObj = Get-AzJson @('cognitiveservices','account','show','-g',$rg,'-n',$foundryAccount)
S 3 'Foundry account'     ($null -ne $fdyObj) $foundryAccount

$mdlObj = Get-AzJson @('cognitiveservices','account','deployment','show','-g',$rg,'-n',$foundryAccount,'--deployment-name',$modelDeployment)
S 3 'Model deployment'    ($null -ne $mdlObj) $modelDeployment

$appReg = (Get-AzJson @('ad','app','list','--display-name',$mcpAppName)) | Select-Object -First 1
S 3 'MCP Entra app'       ($null -ne $appReg) $(if ($appReg) { $appReg.appId } else { $mcpAppName })

$acrObj = Get-AzJson @('acr','show','--name',$acrName)
S 5 'Container registry'  ($null -ne $acrObj) $(if ($acrObj) { "$($acrObj.sku.name) - public=$($acrObj.publicNetworkAccess)" } else { $acrName })

$envObj = Get-AzJson @('containerapp','env','show','-g',$rg,'-n',$envName)
S 6 'Container Apps env'  ($null -ne $envObj) $(if ($envObj) { "vnet-injected=$([bool]$envObj.properties.vnetConfiguration.infrastructureSubnetId)" } else { $envName })

$appObj = Get-AzJson @('containerapp','show','-g',$rg,'-n',$appName)
S 6 'Container app'       ($null -ne $appObj) $(if ($appObj) { $appObj.properties.configuration.ingress.fqdn } else { $appName })

$state | Format-Table -AutoSize -Wrap
$missing = @($state | Where-Object { $_.Status -eq 'missing' }).Count
Write-Host ''
Write-Host ("{0} resource(s) still to create. Cells for anything that exists will skip." -f $missing) -ForegroundColor Cyan

---
## Phase 1 — Foundation

Resource group, monitoring, identity and network. Skips anything that already exists.

### 1.1 — Resource group

In [ ]:
if ((az group exists --name $rg) -eq 'true') { Write-Reused "resource group $rg" }
else {
    az group create --name $rg --location $location -o none
    Write-Created "resource group $rg"
}
if ($vnetRg -ne $rg -and (az group exists --name $vnetRg) -ne 'true') {
    az group create --name $vnetRg --location $location -o none
    Write-Created "resource group $vnetRg"
}

### 1.2 — Log Analytics and Application Insights

The Container Apps environment writes logs here, and Foundry uses App Insights for agent tracing —
which is how you see tool invocations later.

In [ ]:
$law = Get-AzJson @('monitor','log-analytics','workspace','show','-g',$rg,'-n',$lawName)
if ($law) { Write-Reused "Log Analytics $lawName" }
else {
    az monitor log-analytics workspace create -g $rg -n $lawName -l $location --retention-time 30 -o none
    Write-Created "Log Analytics $lawName"
    $law = Get-AzJson @('monitor','log-analytics','workspace','show','-g',$rg,'-n',$lawName)
}

$ai = Get-AzJson @('monitor','app-insights','component','show','-g',$rg,'-a',$appInsightsName)
if ($ai) { Write-Reused "App Insights $appInsightsName" }
else {
    az monitor app-insights component create -g $rg -a $appInsightsName -l $location --workspace $law.id --kind web -o none
    Write-Created "App Insights $appInsightsName"
}

### 1.3 — MCP managed identity

One identity for both pulling the image and reading SQL. No credentials anywhere.

In [ ]:
$uami = Get-AzJson @('identity','show','-g',$rg,'-n',$identityName)
if ($uami) { Write-Reused "identity $identityName" }
else {
    az identity create -g $rg -n $identityName -l $location -o none
    Write-Created "identity $identityName"
    Start-Sleep -Seconds 5
    $uami = Get-AzJson @('identity','show','-g',$rg,'-n',$identityName)
}
Write-Host ("  clientId    : {0}" -f $uami.clientId)
Write-Host ("  principalId : {0}" -f $uami.principalId)

### 1.4 — Virtual network and subnets

Creates the VNet when absent. When it already exists — the usual case where SQL MI is already
deployed — only the two new subnets are added.

**The SQL MI subnet is never modified.** It is delegated to `Microsoft.Sql/managedInstances` and
cannot host a private endpoint or a Container Apps environment. Selecting it produces
*"The selected subnet has a delegation and cannot be used"*.

In [ ]:
$vnet = Get-AzJson @('network','vnet','show','-g',$vnetRg,'-n',$vnetName)
$createMiSubnet = $false

if (-not $vnet) {
    az network vnet create -g $vnetRg -n $vnetName -l $location --address-prefixes $vnetPrefix -o none
    Write-Created "virtual network $vnetName ($vnetPrefix)"
    $createMiSubnet = $true          # greenfield: SQL MI will need a subnet too
    $vnet = Get-AzJson @('network','vnet','show','-g',$vnetRg,'-n',$vnetName)
} else {
    Write-Reused ("virtual network {0} ({1})" -f $vnetName, ($vnet.addressSpace.addressPrefixes -join ', '))
    $existingMi = $vnet.subnets | Where-Object { $_.delegations.serviceName -contains 'Microsoft.Sql/managedInstances' }
    if ($existingMi) { Write-Note ("SQL MI subnet '{0}' present - will not be touched" -f $existingMi.name) }
    else { $createMiSubnet = $true }
}

# -- Container Apps subnet (delegated) --------------------------------------
$sn = Get-AzJson @('network','vnet','subnet','show','-g',$vnetRg,'--vnet-name',$vnetName,'-n',$acaSubnet)
if ($sn) { Write-Reused ("subnet {0} ({1})" -f $acaSubnet, $sn.addressPrefix) }
else {
    az network vnet subnet create -g $vnetRg --vnet-name $vnetName -n $acaSubnet --address-prefixes $acaPrefix -o none
    Write-Created "subnet $acaSubnet ($acaPrefix)"
}
az network vnet subnet update -g $vnetRg --vnet-name $vnetName -n $acaSubnet --delegations Microsoft.App/environments -o none

# -- Private endpoint subnet (NO delegation) --------------------------------
$sn = Get-AzJson @('network','vnet','subnet','show','-g',$vnetRg,'--vnet-name',$vnetName,'-n',$peSubnet)
if ($sn) { Write-Reused ("subnet {0} ({1})" -f $peSubnet, $sn.addressPrefix) }
else {
    az network vnet subnet create -g $vnetRg --vnet-name $vnetName -n $peSubnet --address-prefixes $pePrefix -o none
    Write-Created "subnet $peSubnet ($pePrefix)"
}
az network vnet subnet update -g $vnetRg --vnet-name $vnetName -n $peSubnet --disable-private-endpoint-network-policies true -o none

# -- SQL MI subnet - only when we are also creating the instance ------------
if ($createMiSubnet) {
    $nsgName = "nsg-$miSubnet"
    $rtName  = "rt-$miSubnet"
    if (-not (Get-AzJson @('network','nsg','show','-g',$vnetRg,'-n',$nsgName))) {
        az network nsg create -g $vnetRg -n $nsgName -l $location -o none
        Write-Created "NSG $nsgName"
    }
    if (-not (Get-AzJson @('network','route-table','show','-g',$vnetRg,'-n',$rtName))) {
        az network route-table create -g $vnetRg -n $rtName -l $location -o none
        Write-Created "route table $rtName"
    }
    $sn = Get-AzJson @('network','vnet','subnet','show','-g',$vnetRg,'--vnet-name',$vnetName,'-n',$miSubnet)
    if ($sn) { Write-Reused "subnet $miSubnet" }
    else {
        az network vnet subnet create -g $vnetRg --vnet-name $vnetName -n $miSubnet `
            --address-prefixes $miPrefix --network-security-group $nsgName --route-table $rtName -o none
        az network vnet subnet update -g $vnetRg --vnet-name $vnetName -n $miSubnet `
            --delegations Microsoft.Sql/managedInstances -o none
        Write-Created "subnet $miSubnet ($miPrefix) delegated to SQL MI"
    }
}

# -- Private DNS zone for the registry --------------------------------------
$zone = 'privatelink.azurecr.io'
if (Get-AzJson @('network','private-dns','zone','show','-g',$vnetRg,'-n',$zone)) { Write-Reused "DNS zone $zone" }
else { az network private-dns zone create -g $vnetRg -n $zone -o none; Write-Created "DNS zone $zone" }

$linkName = "link-$vnetName"
if (Get-AzJson @('network','private-dns','link','vnet','show','-g',$vnetRg,'--zone-name',$zone,'-n',$linkName)) {
    Write-Reused "DNS link $linkName"
} else {
    az network private-dns link vnet create -g $vnetRg --zone-name $zone -n $linkName `
        --virtual-network $vnetName --registration-enabled false -o none
    Write-Created "DNS link $linkName"
}

# -- Verify -----------------------------------------------------------------
Write-Host ''
$a = Get-AzJson @('network','vnet','subnet','show','-g',$vnetRg,'--vnet-name',$vnetName,'-n',$acaSubnet)
$p = Get-AzJson @('network','vnet','subnet','show','-g',$vnetRg,'--vnet-name',$vnetName,'-n',$peSubnet)
@(
  [pscustomobject]@{ Subnet=$acaSubnet; Prefix=$a.addressPrefix; Delegation=$a.delegations[0].serviceName; Policies='n/a' }
  [pscustomobject]@{ Subnet=$peSubnet;  Prefix=$p.addressPrefix; Delegation=$(if ($p.delegations) { $p.delegations[0].serviceName } else { 'none' }); Policies=$p.privateEndpointNetworkPolicies }
) | Format-Table -AutoSize

$ok = ($a.delegations[0].serviceName -eq 'Microsoft.App/environments') -and (-not $p.delegations) -and ($p.privateEndpointNetworkPolicies -eq 'Disabled')
if ($ok) { Write-Host 'PHASE 1 PASS' -ForegroundColor Green } else { Write-Host 'PHASE 1 FAIL - check the table above' -ForegroundColor Red }

$dns = Get-AzJson @('network','vnet','show','-g',$vnetRg,'-n',$vnetName,'--query','dhcpOptions.dnsServers')
if ($dns -and $dns.Count -gt 0) {
    Write-Note ("Custom DNS: {0} - these must forward privatelink.azurecr.io to 168.63.129.16" -f ($dns -join ', '))
}

---
## Phase 2 — SQL Managed Instance

> [!WARNING]
> **Creating a managed instance takes 4 to 6 hours.** Cell 2.1 starts it and returns immediately.
> Cell 2.2 reports progress and can be re-run whenever you come back. Phases 3 and 5 do not depend
> on SQL MI, so continue with those while it provisions.
>
> If you already have an instance, 2.1 detects it and skips.

### 2.1 — Create or detect the instance

In [ ]:
$mi = Get-SqlMi
if ($mi) {
    Write-Reused ("SQL MI {0} - state={1}" -f $miName, $mi.state)
    $miFqdn = $mi.fullyQualifiedDomainName
    Write-Host ("  VNet-local FQDN : {0}" -f $miFqdn)
    Write-Host ("  public endpoint : {0}" -f $mi.publicDataEndpointEnabled)
} else {
    $miSubnetId = az network vnet subnet show -g $vnetRg --vnet-name $vnetName -n $miSubnet --query id -o tsv
    if (-not $miSubnetId) { throw "Subnet $miSubnet not found. Re-run cell 1.4." }

    if ([string]::IsNullOrWhiteSpace($miEntraAdmin)) { $miEntraAdmin = $acct.user.name }
    $adminObjId = az ad signed-in-user show --query id -o tsv

    Write-Note 'Starting SQL MI provisioning - this takes 4 to 6 hours.'
    az sql mi create -g $rg -n $miName -l $location `
        --subnet $miSubnetId `
        --capacity $miVCores --storage $miStorageGB `
        --edition GeneralPurpose --family Gen5 --license-type LicenseIncluded `
        --enable-ad-only-auth --external-admin-principal-type User `
        --external-admin-name $miEntraAdmin --external-admin-sid $adminObjId `
        --public-data-endpoint false `
        --no-wait -o none
    Write-Created "SQL MI $miName (provisioning started)"
    Write-Host '  Continue with Phases 3 and 5 while this runs. Check progress with cell 2.2.' -ForegroundColor Cyan
}

### 2.2 — Check provisioning progress

Read-only. Re-run whenever you come back.

In [ ]:
$mi = Get-SqlMi
if (-not $mi) { Write-Failed "SQL MI $miName not found - run cell 2.1" }
else {
    [pscustomobject]@{
        Name = $mi.name; State = $mi.state
        FQDN = $mi.fullyQualifiedDomainName
        Subnet = ($mi.subnetId -split '/')[-1]
        PublicEndpoint = $mi.publicDataEndpointEnabled
    } | Format-List

    $miFqdn = $mi.fullyQualifiedDomainName
    if ($mi.state -eq 'Ready') { Write-Host 'SQL MI is ready.' -ForegroundColor Green }
    else { Write-Note ("Still provisioning (state={0}). Re-run this cell later." -f $mi.state) }
}

### 2.3 — Entra Directory Readers

Creating the MCP database user needs the **Directory Readers** directory role on the managed
instance's server identity. It cannot be granted from the CLI here — it needs a Privileged Role
Administrator or Global Administrator, and usually follows a slower approval path than Azure RBAC.

**Request it now**, not when Phase 4 fails.

In [ ]:
$mi = Get-SqlMi
if ($mi -and $mi.identity -and $mi.identity.principalId) {
    Write-Host 'Ask a Privileged Role Administrator or Global Administrator to grant:' -ForegroundColor Cyan
    Write-Host ''
    Write-Host  '  Role      : Directory Readers'
    Write-Host ("  To        : {0} (SQL MI server identity)" -f $mi.name)
    Write-Host ("  Object ID : {0}" -f $mi.identity.principalId)
    Write-Host ''
    Write-Host 'Entra admin center > Roles and administrators > Directory Readers > Add assignment'
} else {
    Write-Note 'SQL MI has no system-assigned identity yet - re-run once provisioning completes.'
}

---
## Phase 3 — Foundry and Entra

Independent of SQL MI, so run this while the instance provisions.

### 3.1 — Foundry account, project and model

In [ ]:
$fdy = Get-AzJson @('cognitiveservices','account','show','-g',$rg,'-n',$foundryAccount)
if ($fdy) { Write-Reused "Foundry account $foundryAccount" }
else {
    az cognitiveservices account create -g $rg -n $foundryAccount -l $location `
        --kind AIServices --sku S0 --custom-domain $foundryAccount --assign-identity --yes -o none
    Write-Created "Foundry account $foundryAccount"
}

$mdl = Get-AzJson @('cognitiveservices','account','deployment','show','-g',$rg,'-n',$foundryAccount,'--deployment-name',$modelDeployment)
if ($mdl) { Write-Reused "model deployment $modelDeployment" }
else {
    az cognitiveservices account deployment create -g $rg -n $foundryAccount `
        --deployment-name $modelDeployment --model-name $modelName --model-version $modelVersion `
        --model-format OpenAI --sku-name GlobalStandard --sku-capacity $modelCapacity -o none
    Write-Created "model deployment $modelDeployment"
}

# Project - created via ARM; there is no first-class CLI verb
$projUri = "/subscriptions/$subscriptionId/resourceGroups/$rg/providers/Microsoft.CognitiveServices/accounts/$foundryAccount/projects/$foundryProject" + "?api-version=2025-04-01-preview"
$proj = Get-AzJson @('rest','--method','get','--url',$projUri)
if ($proj) { Write-Reused "Foundry project $foundryProject" }
else {
    $body = @{ location = $location; identity = @{ type = 'SystemAssigned' }
               properties = @{ displayName = $foundryProject; description = 'SQL MCP project' } } | ConvertTo-Json -Depth 10
    $tmp = New-TemporaryFile
    Set-Content -LiteralPath $tmp -Value $body -Encoding utf8NoBOM
    az rest --method put --url $projUri --headers 'Content-Type=application/json' --body "@$tmp" -o none
    Remove-Item $tmp -Force
    Write-Created "Foundry project $foundryProject"
    Start-Sleep -Seconds 20
    $proj = Get-AzJson @('rest','--method','get','--url',$projUri)
}
$projectPrincipalId = $proj.identity.principalId
Write-Host ("  project principalId : {0}" -f $projectPrincipalId)

### 3.2 — Entra application and the `Mcp.Invoke` role

This is what lets Foundry obtain a token DAB will accept. The application role must be assigned to
the **project's** managed identity — the portal cannot do this reliably, which is why it is
scripted.

In [ ]:
if (-not $repoRoot) { throw 'Repository root not found - re-run cell 0.2.' }

Push-Location $repoRoot
try {
    & ./scripts/setup-demo-entra.ps1 -ProjectPrincipalId $projectPrincipalId -TenantId $tenantId -ApplicationDisplayName $mcpAppName
} finally { Pop-Location }

$appReg = (Get-AzJson @('ad','app','list','--display-name',$mcpAppName)) | Select-Object -First 1
$mcpAppId = $appReg.appId
Write-Host ''
Write-Host ("  MCP app client ID : {0}" -f $mcpAppId) -ForegroundColor Green
Write-Host ("  DAB audience      : {0}   (bare GUID)" -f $mcpAppId)
Write-Host ("  Foundry audience  : api://{0}" -f $mcpAppId)
Write-Note 'These two audience forms are intentionally different. Mixing them causes a 401 later.'

# The application must emit v2 tokens. Left at the Entra default it emits v1, whose aud is
# "api://<id>" and whose issuer is sts.windows.net - neither of which DAB accepts. The role
# assignment is still correct and the token still carries Mcp.Invoke, so nothing upstream
# looks wrong and the failure only appears when the agent calls a tool.
$tokenVersion = az ad app show --id $mcpAppId --query 'api.requestedAccessTokenVersion' -o tsv
if ($tokenVersion -eq '2') {
    Write-Host '  access token version : 2' -ForegroundColor Green
} else {
    Write-Failed ("Access token version is '{0}', must be 2. Entra would issue v1 tokens and every agent call would return 401." -f $tokenVersion)
    throw 'Fix the access token version before continuing.'
}

---
## Phase 4 — The data contract

**This phase determines whether the agent gives good answers.**

The agent never sees your tables. It sees the entities described in `dab-config.json`, backed by
curated views that SQL grants access to. Both halves must agree, and the container validates them
at startup — so getting this right here avoids a failure two phases later that looks like a
permissions problem.

Full reasoning: [`docs/configure-for-your-database.md`](../docs/configure-for-your-database.md).

### 4.1 — Choose the views to expose

List the views the agent may read. Start with **three to five**. More entities makes answers worse,
not better — every one without a good description is another chance to guess wrong.

In [ ]:
# Views the agent may read, in the mcp schema (name only, no schema prefix).
$curatedViews = @(
    # 'vw_example_summary'
    # 'vw_example_detail'
)

# Stored procedures exposed as named tools (optional).
$curatedProcs = @(
    # 'usp_GetExampleById'
)

if ($curatedViews.Count -eq 0 -and $curatedProcs.Count -eq 0) {
    Write-Note 'No objects listed yet. Add view names above before continuing.'
    Write-Host ''
    Write-Host 'To see what already exists, run this against the instance:' -ForegroundColor Cyan
    Write-Host "  SELECT s.name + '.' + v.name FROM sys.views v JOIN sys.schemas s ON s.schema_id = v.schema_id ORDER BY 1;"
} else {
    Write-Host ("Views : {0}" -f ($curatedViews -join ', '))
    Write-Host ("Procs : {0}" -f $(if ($curatedProcs.Count) { $curatedProcs -join ', ' } else { 'none' }))
}

### 4.2 — Generate the SQL contract

Creates the `mcp` schema, the `mcp_reader` role, the MCP database user, and grants on **named
objects only**. Never `db_datareader` — that would expose every table, including ones added later.

Review the generated file before applying it.

In [ ]:
$sb = [System.Text.StringBuilder]::new()
[void]$sb.AppendLine("-- SQL contract for the MCP agent")
[void]$sb.AppendLine("-- Target database: [$databaseName]")
[void]$sb.AppendLine("")
[void]$sb.AppendLine("USE [$databaseName];")
[void]$sb.AppendLine("GO")
[void]$sb.AppendLine("")
[void]$sb.AppendLine("IF SCHEMA_ID(N'mcp') IS NULL EXEC(N'CREATE SCHEMA mcp AUTHORIZATION dbo;');")
[void]$sb.AppendLine("GO")
[void]$sb.AppendLine("")
[void]$sb.AppendLine("IF DATABASE_PRINCIPAL_ID(N'mcp_reader') IS NULL")
[void]$sb.AppendLine("    CREATE ROLE [mcp_reader] AUTHORIZATION [dbo];")
[void]$sb.AppendLine("GO")
[void]$sb.AppendLine("")
[void]$sb.AppendLine("-- MCP managed identity as a contained database user.")
[void]$sb.AppendLine("-- Requires Entra Directory Readers on the SQL MI server identity (cell 2.3).")
[void]$sb.AppendLine("IF DATABASE_PRINCIPAL_ID(N'$identityName') IS NULL")
[void]$sb.AppendLine("    CREATE USER [$identityName] FROM EXTERNAL PROVIDER;")
[void]$sb.AppendLine("GO")
[void]$sb.AppendLine("")
[void]$sb.AppendLine("IF NOT EXISTS (SELECT 1 FROM sys.database_role_members rm")
[void]$sb.AppendLine("    JOIN sys.database_principals r ON r.principal_id = rm.role_principal_id")
[void]$sb.AppendLine("    JOIN sys.database_principals m ON m.principal_id = rm.member_principal_id")
[void]$sb.AppendLine("    WHERE r.name = N'mcp_reader' AND m.name = N'$identityName')")
[void]$sb.AppendLine("    ALTER ROLE [mcp_reader] ADD MEMBER [$identityName];")
[void]$sb.AppendLine("GO")
[void]$sb.AppendLine("")

if ($curatedViews.Count -eq 0 -and $curatedProcs.Count -eq 0) {
    [void]$sb.AppendLine("-- No objects listed in cell 4.1. Example shape:")
    [void]$sb.AppendLine("-- CREATE OR ALTER VIEW mcp.vw_example AS")
    [void]$sb.AppendLine("--     SELECT t.Id, t.Name, t.Status FROM dbo.SourceTable AS t WHERE t.IsDeleted = 0;")
    [void]$sb.AppendLine("-- GO")
    [void]$sb.AppendLine("-- GRANT SELECT ON OBJECT::mcp.vw_example TO [mcp_reader];")
    [void]$sb.AppendLine("-- GO")
} else {
    [void]$sb.AppendLine("-- Grants on named objects only.")
    foreach ($v in $curatedViews) { [void]$sb.AppendLine("GRANT SELECT  ON OBJECT::mcp.$v TO [mcp_reader];") }
    foreach ($p in $curatedProcs) { [void]$sb.AppendLine("GRANT EXECUTE ON OBJECT::mcp.$p TO [mcp_reader];") }
    [void]$sb.AppendLine("GO")
}

$genDir = Join-Path $repoRoot 'notebooks/generated'
New-Item -ItemType Directory -Force -Path $genDir | Out-Null
$sqlPath = Join-Path $genDir 'mcp-contract.sql'
Set-Content -LiteralPath $sqlPath -Value $sb.ToString() -Encoding utf8NoBOM
Write-Created "SQL contract -> $sqlPath"
Write-Host ''
Get-Content $sqlPath | Select-Object -First 40

### 4.3 — Apply the contract

Runs the script against the managed instance using your Entra identity.

The client must be able to reach the instance. If you are outside the VNet and the public endpoint
is still enabled, this works over 3342; otherwise run it from a host inside the network. The cell
reports which path it used.

In [ ]:
$sqlcmd = (Get-Command sqlcmd -ErrorAction SilentlyContinue).Source
if (-not $sqlcmd) {
    Write-Failed 'sqlcmd not found. Install it: winget install --id Microsoft.Sqlcmd'
} else {
    $mi = Get-SqlMi
    if ($mi.publicDataEndpointEnabled) {
        $target = ($mi.fullyQualifiedDomainName -replace '\.database\.windows\.net$', '.public.database.windows.net') + ',3342'
        Write-Note "Using the public endpoint ($target) - available because it is still enabled."
    } else {
        $target = $mi.fullyQualifiedDomainName + ',1433'
        Write-Note "Using the VNet-local endpoint ($target) - you must be inside the network."
    }

    & $sqlcmd -S $target -d $databaseName --authentication-method ActiveDirectoryAzCli -N true -b -I -i $sqlPath
    if ($LASTEXITCODE -eq 0) { Write-Host 'Contract applied.' -ForegroundColor Green }
    else { Write-Failed "sqlcmd exited $LASTEXITCODE - see the error above." }
}

> **A principal-resolution error** means Directory Readers has not been granted — see cell 2.3.
>
> **If a view reads across databases**, a contained user is not enough. SQL MI needs a server-level
> login:
>
> ```sql
> USE master;
> CREATE LOGIN [<uami-name>] FROM EXTERNAL PROVIDER;
> -- then in EVERY database the view touches:
> DROP USER IF EXISTS [<uami-name>];
> CREATE USER [<uami-name>] FROM LOGIN [<uami-name>];
> ALTER ROLE [mcp_reader] ADD MEMBER [<uami-name>];
> ```
>
> The `DROP USER` matters — an existing contained user blocks the login-backed one.

### 4.4 — Verify the grants

Expect only your curated objects, and exactly one role member.

In [ ]:
$verify = @"
SET NOCOUNT ON;
SELECT s.name AS [schema], o.name AS [object], p.permission_name
FROM sys.database_permissions p
JOIN sys.database_principals dp ON dp.principal_id = p.grantee_principal_id
JOIN sys.objects o ON o.object_id = p.major_id
JOIN sys.schemas s ON s.schema_id = o.schema_id
WHERE dp.name = N'mcp_reader' ORDER BY s.name, o.name;

SELECT m.name AS role_member
FROM sys.database_role_members rm
JOIN sys.database_principals r ON r.principal_id = rm.role_principal_id
JOIN sys.database_principals m ON m.principal_id = rm.member_principal_id
WHERE r.name = N'mcp_reader';
"@
$vPath = Join-Path $genDir 'verify-grants.sql'
Set-Content -LiteralPath $vPath -Value $verify -Encoding utf8NoBOM

if ($sqlcmd) { & $sqlcmd -S $target -d $databaseName --authentication-method ActiveDirectoryAzCli -N true -b -I -i $vPath }
else { Write-Failed 'sqlcmd not available' }

### 4.5 — Generate the DAB entities

**Without this step nothing downstream works.** The committed `dab-config.json` describes the
demo's synthetic views. Building an image from it and pointing it at your database makes the
container fail at startup with `Cannot obtain schema for entity`.

This reads the real column names from your views and writes matching entities.

> Field **descriptions** drive answer quality more than anything else. This writes placeholders —
> edit them before the agent goes near a user. Where a column holds a fixed set of values, list
> them: without `Valid values: ACTIVE, CLOSED`, an agent asked for closed records will filter on
> `'Closed'`, get nothing, and confidently report there are none.

In [ ]:
if ($curatedViews.Count -eq 0) { throw 'List your views in cell 4.1 first.' }

$colQuery = @"
SET NOCOUNT ON;
SELECT TABLE_NAME + '|' + COLUMN_NAME + '|' + DATA_TYPE
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'mcp'
ORDER BY TABLE_NAME, ORDINAL_POSITION;
"@
$cPath = Join-Path $genDir 'columns.sql'
Set-Content -LiteralPath $cPath -Value $colQuery -Encoding utf8NoBOM
$raw = & $sqlcmd -S $target -d $databaseName --authentication-method ActiveDirectoryAzCli -N true -b -I -h -1 -W -i $cPath

$byView = @{}
foreach ($line in $raw) {
    if ([string]::IsNullOrWhiteSpace($line)) { continue }
    if ($line -match 'rows affected') { continue }
    $parts = $line.Trim() -split '\|'
    if ($parts.Count -lt 3) { continue }
    if (-not $byView.ContainsKey($parts[0])) { $byView[$parts[0]] = @() }
    $byView[$parts[0]] += [pscustomobject]@{ Name = $parts[1]; Type = $parts[2] }
}

function ConvertTo-EntityName($raw, $prefix) {
    $s = $raw -replace ('^' + $prefix), ''
    return (($s -split '_') | Where-Object { $_ } | ForEach-Object { $_.Substring(0,1).ToUpper() + $_.Substring(1) }) -join ''
}

$cfgPath = Join-Path $repoRoot 'src/mcp-server/dab-config.json'
$cfg = Get-Content $cfgPath -Raw | ConvertFrom-Json

$entities = [ordered]@{}
foreach ($v in $curatedViews) {
    $cols = $byView[$v]
    if (-not $cols) { Write-Failed "view mcp.$v returned no columns - does it exist?"; continue }
    $fields = @()
    $first = $true
    foreach ($c in $cols) {
        $f = [ordered]@{ name = $c.Name; description = "TODO describe $($c.Name) ($($c.Type))" }
        if ($first) { $f['primary-key'] = $true; $first = $false }
        $fields += $f
    }
    $entities[(ConvertTo-EntityName $v 'vw_')] = [ordered]@{
        description = "TODO describe what mcp.$v contains and when to use it"
        source      = [ordered]@{ object = "mcp.$v"; type = 'view' }
        fields      = $fields
        permissions = @(
            [ordered]@{ role = 'Mcp.Invoke';    actions = @(@{ action = 'read' }) }
            [ordered]@{ role = 'authenticated'; actions = @(@{ action = 'read' }) })
        mcp         = [ordered]@{ 'dml-tools' = $true; 'custom-tool' = $false }
    }
}
foreach ($p in $curatedProcs) {
    $entities[(ConvertTo-EntityName $p 'usp_')] = [ordered]@{
        description = "TODO describe what mcp.$p returns and when to use it"
        source      = [ordered]@{ object = "mcp.$p"; type = 'stored-procedure'
                                  parameters = @(@{ name = 'TODO'; description = 'TODO'; required = $true }) }
        permissions = @(
            [ordered]@{ role = 'Mcp.Invoke';    actions = @(@{ action = 'execute' }) }
            [ordered]@{ role = 'authenticated'; actions = @(@{ action = 'execute' }) })
        mcp         = [ordered]@{ 'custom-tool' = $true; 'dml-tools' = $false }
    }
}

$backup = "$cfgPath.demo-backup"
if (-not (Test-Path $backup)) { Copy-Item $cfgPath $backup; Write-Note "Demo config backed up to $backup" }

$cfg.entities = $entities
$cfg | ConvertTo-Json -Depth 20 | Set-Content -LiteralPath $cfgPath -Encoding utf8NoBOM
Write-Created ("dab-config.json rewritten with {0} entities" -f $entities.Count)
Write-Host ''
foreach ($k in $entities.Keys) { Write-Host ("  {0} -> {1}" -f $k, $entities[$k].source.object) }
Write-Host ''
Write-Note 'Now edit the TODO descriptions in src/mcp-server/dab-config.json before building.'

### 4.6 — Validate the config against the database

Confirms every entity maps to an object that exists and is granted. **This turns a container
startup failure into a preflight failure**, and it gates Phase 5.

In [ ]:
$cfgPath = Join-Path $repoRoot 'src/mcp-server/dab-config.json'
$cfg = Get-Content $cfgPath -Raw | ConvertFrom-Json

$objects = @()
foreach ($n in $cfg.entities.PSObject.Properties.Name) { $objects += $cfg.entities.$n.source.object }

Write-Host 'Entities in dab-config.json:'
$objects | ForEach-Object { Write-Host "  $_" }
Write-Host ''

$demoPattern = 'vw_transfer_|vw_client_account|vw_advisor_|usp_GetTransfer|usp_GetOpen|usp_GetAdvisor'
$demoLeft = @($objects | Where-Object { $_ -match $demoPattern })

if ($demoLeft.Count -gt 0) {
    Write-Failed 'Config still references demo objects - Phase 5 is blocked:'
    $demoLeft | ForEach-Object { Write-Host "    $_" -ForegroundColor Red }
    $configReady = $false
} else {
    $lines = foreach ($o in $objects) {
        $s, $n = $o -split '\.'
        "IF OBJECT_ID(N'$s.$n') IS NULL PRINT 'MISSING $s.$n'; ELSE IF NOT EXISTS (SELECT 1 FROM sys.database_permissions p JOIN sys.database_principals dp ON dp.principal_id = p.grantee_principal_id WHERE dp.name='mcp_reader' AND p.major_id = OBJECT_ID(N'$s.$n')) PRINT 'NO GRANT $s.$n'; ELSE PRINT 'OK $s.$n';"
    }
    $checkSql = "SET NOCOUNT ON;`n" + ($lines -join "`n")
    $kPath = Join-Path $genDir 'check-objects.sql'
    Set-Content -LiteralPath $kPath -Value $checkSql -Encoding utf8NoBOM

    $out = & $sqlcmd -S $target -d $databaseName --authentication-method ActiveDirectoryAzCli -N true -b -I -h -1 -W -i $kPath
    $rows = $out | Where-Object { $_ -match '^(OK|MISSING|NO GRANT)' }
    foreach ($r in $rows) {
        $colour = if ($r -match '^OK') { 'Green' } else { 'Red' }
        Write-Host ("  {0}" -f $r) -ForegroundColor $colour
    }
    $bad = @($rows | Where-Object { $_ -match '^(MISSING|NO GRANT)' })
    $configReady = ($bad.Count -eq 0 -and $rows.Count -gt 0)
}

Write-Host ''
if ($configReady) { Write-Host 'PHASE 4 PASS - config and database agree.' -ForegroundColor Green }
else { Write-Host 'PHASE 4 FAIL - fix before building the image.' -ForegroundColor Red }

---
## Phase 5 — Container registry

**Order matters.** The image is built while the registry is still publicly reachable, and public
access is disabled afterwards. `az acr build` stops working once public access is off — ACR Tasks
need public IPs unless you assign a dedicated agent pool.

### 5.1 — Premium registry and `AcrPull`

Premium is required: private endpoints are a Premium-tier feature.

In [ ]:
$acr = Get-AzJson @('acr','show','--name',$acrName)
if ($acr) {
    Write-Reused ("registry {0} (sku {1})" -f $acrName, $acr.sku.name)
    if ($acr.sku.name -ne 'Premium') { az acr update --name $acrName --sku Premium -o none; Write-Created 'upgraded to Premium' }
} else {
    az acr create --name $acrName -g $rg -l $location --sku Premium -o none
    Write-Created "registry $acrName (Premium)"
}
az acr config authentication-as-arm update -r $acrName --status enabled -o none

$acrId = az acr show --name $acrName --query id -o tsv
$loginServer = az acr show --name $acrName --query loginServer -o tsv
$uami = Get-AzJson @('identity','show','-g',$rg,'-n',$identityName)

$ra = Get-AzJson @('role','assignment','list','--assignee',$uami.principalId,'--scope',$acrId,'--role','AcrPull')
if ($ra -and $ra.Count -gt 0) { Write-Reused 'AcrPull assignment' }
else {
    az role assignment create --assignee-object-id $uami.principalId --assignee-principal-type ServicePrincipal `
        --role AcrPull --scope $acrId -o none
    Write-Created 'AcrPull assignment'
    Start-Sleep -Seconds 10
}
Write-Host ("  loginServer : {0}" -f $loginServer)

### 5.2 — Build and push the image

Gated on Phase 4: the build is refused while the config describes objects that do not exist in your
database.

In [ ]:
if (-not $configReady) {
    Write-Failed 'Phase 4 has not passed. Run cell 4.6 and resolve the findings before building.'
} else {
    $acrPublic = az acr show --name $acrName --query publicNetworkAccess -o tsv
    if ($acrPublic -eq 'Disabled') {
        Write-Failed 'Registry public access is already disabled - az acr build will not work.'
        Write-Host ("  Re-enable temporarily:  az acr update --name {0} --public-network-enabled true" -f $acrName) -ForegroundColor Cyan
    } else {
        Push-Location $repoRoot
        try {
            & ./scripts/build-demo-mcp.ps1 -RegistryName $acrName -McpApplicationId $mcpAppId -TenantId $tenantId
        } finally { Pop-Location }
        Write-Host ''
        az acr repository show-tags --name $acrName --repository sql-mcp -o table
    }
}

### 5.3 — Private endpoint, DNS records, then lock down

Configuring a private endpoint auto-enables *dedicated data endpoints*, so DNS needs a record for
the registry **and** one per region for the data endpoint. The `dns-zone-group` command creates
both — creating records by hand and missing the data endpoint produces image pulls that
authenticate and then hang.

In [ ]:
$peName = "pe-$acrName"
$acrId  = az acr show --name $acrName --query id -o tsv

if (Get-AzJson @('network','private-endpoint','show','-g',$vnetRg,'-n',$peName)) { Write-Reused "private endpoint $peName" }
else {
    az network private-endpoint create -n $peName -g $vnetRg --vnet-name $vnetName --subnet $peSubnet `
        --private-connection-resource-id $acrId --group-ids registry --connection-name "conn-$acrName" -o none
    Write-Created "private endpoint $peName"
}

$zoneId = az network private-dns zone show -g $vnetRg -n 'privatelink.azurecr.io' --query id -o tsv
if (Get-AzJson @('network','private-endpoint','dns-zone-group','show','--endpoint-name',$peName,'-g',$vnetRg,'-n','default')) {
    Write-Reused 'DNS zone group'
} else {
    az network private-endpoint dns-zone-group create -g $vnetRg --endpoint-name $peName `
        --name default --private-dns-zone $zoneId --zone-name acr -o none
    Write-Created 'DNS zone group'
}

Write-Host ''
Write-Host 'A records (expect the registry AND a .data. record):'
az network private-dns record-set a list -g $vnetRg --zone-name 'privatelink.azurecr.io' `
    --query "[].{name:name, ip:aRecords[0].ipv4Address}" -o table

$tags = Get-AzJson @('acr','repository','show-tags','--name',$acrName,'--repository','sql-mcp')
Write-Host ''
if (-not $tags -or $tags.Count -eq 0) {
    Write-Failed 'No image tags - not disabling public access. Run cell 5.2 first.'
} else {
    az acr update --name $acrName --public-network-enabled false -o none
    Write-Created ("public access disabled ({0} tag(s) present)" -f $tags.Count)
}

---
## Phase 6 — Container Apps

Ingress stays **external**. Foundry is public and reaches the MCP endpoint over the internet with
an Entra token — the network is not the trust boundary on that hop.

### 6.1 — VNet-injected environment

Takes several minutes.

In [ ]:
$acaEnv = Get-AzJson @('containerapp','env','show','-g',$rg,'-n',$envName)
if ($acaEnv) { Write-Reused "environment $envName" }
else {
    $acaSubnetId = az network vnet subnet show -g $vnetRg --vnet-name $vnetName -n $acaSubnet --query id -o tsv
    $lawCustomerId = az monitor log-analytics workspace show -g $rg -n $lawName --query customerId -o tsv
    $lawKey = az monitor log-analytics workspace get-shared-keys -g $rg -n $lawName --query primarySharedKey -o tsv

    Write-Note 'Creating VNet-injected environment - several minutes.'
    az containerapp env create -n $envName -g $rg -l $location `
        --infrastructure-subnet-resource-id $acaSubnetId --enable-workload-profiles `
        --logs-destination log-analytics --logs-workspace-id $lawCustomerId --logs-workspace-key $lawKey -o none
    Write-Created "environment $envName"
    $acaEnv = Get-AzJson @('containerapp','env','show','-g',$rg,'-n',$envName)
}
[pscustomobject]@{
    Name = $acaEnv.name; State = $acaEnv.properties.provisioningState
    Subnet = ($acaEnv.properties.vnetConfiguration.infrastructureSubnetId -split '/')[-1]
    Internal = $acaEnv.properties.vnetConfiguration.internal
} | Format-List

### 6.2 — Container app

In [ ]:
$loginServer = az acr show --name $acrName --query loginServer -o tsv
$identityId  = az identity show -g $rg -n $identityName --query id -o tsv
$uamiClient  = az identity show -g $rg -n $identityName --query clientId -o tsv

$mi = Get-SqlMi
$miFqdn = $mi.fullyQualifiedDomainName

$imageTag = az acr repository show-tags --name $acrName --repository sql-mcp --top 1 --orderby time_desc -o tsv
if (-not $imageTag) { throw 'No image in the registry. Run cell 5.2.' }
$image = "$loginServer/sql-mcp:$imageTag"
$connStr = "Server=tcp:$miFqdn,1433;Initial Catalog=$databaseName;Authentication=Active Directory Managed Identity;User Id=$uamiClient;Encrypt=True;TrustServerCertificate=False;Connection Timeout=30;"

Write-Host "Image : $image"
Write-Host "SQL   : $miFqdn,1433 / $databaseName"

$app = Get-AzJson @('containerapp','show','-g',$rg,'-n',$appName)
if ($app) {
    az containerapp update -n $appName -g $rg --image $image `
        --set-env-vars "DAB_ENVIRONMENT=Production" "DATABASE_CONNECTION_STRING=$connStr" -o none
    Write-Created "revision updated to $imageTag"
} else {
    az containerapp create -n $appName -g $rg --environment $envName `
        --user-assigned $identityId --registry-identity $identityId `
        --registry-server $loginServer --image $image `
        --target-port 5000 --ingress external --transport http `
        --cpu 0.5 --memory 1.0Gi --min-replicas 1 --max-replicas 1 `
        --env-vars "DAB_ENVIRONMENT=Production" "DATABASE_CONNECTION_STRING=$connStr" -o none
    Write-Created "container app $appName"
}
$app = Get-AzJson @('containerapp','show','-g',$rg,'-n',$appName)
$mcpFqdn = $app.properties.configuration.ingress.fqdn
Write-Host ("  MCP endpoint : https://{0}/mcp" -f $mcpFqdn) -ForegroundColor Green

### 6.3 — Verify the revision

In [ ]:
Start-Sleep -Seconds 20
$revs = Get-AzJson @('containerapp','revision','list','-n',$appName,'-g',$rg)
$latest = $revs | Sort-Object { $_.properties.createdTime } -Descending | Select-Object -First 1
[pscustomobject]@{ Revision = $latest.name; Active = $latest.properties.active; State = $latest.properties.runningState } | Format-List

if ($latest.properties.runningState -ne 'Running') {
    Write-Note 'Not running - last 60 log lines:'
    az containerapp logs show -n $appName -g $rg --tail 60
    Write-Host ''
    Write-Host 'Common causes:' -ForegroundColor Yellow
    Write-Host '  "is not able to access the database"  -> cross-database view needs a server-level login (4.3 note)'
    Write-Host '  "Login failed"                        -> identity not in mcp_reader, or Directory Readers missing'
    Write-Host '  "Cannot obtain schema for entity"     -> object missing or not granted (re-run 4.6)'
    Write-Host '  connection timeout                    -> DNS or routing to the MI VNet-local FQDN'
} else { Write-Host 'PHASE 6 PASS' -ForegroundColor Green }

---
## Phase 7 — Foundry connection and agent

The two audience forms are **intentionally different**: the Foundry connection uses
`api://<client-id>`; DAB validates the bare GUID.

### 7.1 — Project MCP connection

In [ ]:
$connName = 'sql-mcp'
$connUri = "/subscriptions/$subscriptionId/resourceGroups/$rg/providers/Microsoft.CognitiveServices/accounts/$foundryAccount/projects/$foundryProject/connections/$connName" + "?api-version=2025-04-01-preview"

$existing = Get-AzJson @('rest','--method','get','--url',$connUri)
if ($existing) { Write-Reused "connection $connName" }
else {
    $body = @{ properties = @{
        category                    = 'RemoteTool'
        authType                    = 'ProjectManagedIdentity'
        target                      = "https://$mcpFqdn/mcp"
        isSharedToAll               = $true
        useWorkspaceManagedIdentity = $true
        audience                    = "api://$mcpAppId"
        metadata                    = @{ ApiType = 'Azure' }
    }} | ConvertTo-Json -Depth 10
    $tmp = New-TemporaryFile
    Set-Content -LiteralPath $tmp -Value $body -Encoding utf8NoBOM
    az rest --method put --url $connUri --headers 'Content-Type=application/json' --body "@$tmp" -o none
    Remove-Item $tmp -Force
    Write-Created "connection $connName"
    $existing = Get-AzJson @('rest','--method','get','--url',$connUri)
}
[pscustomobject]@{
    Name = $connName; Category = $existing.properties.category
    AuthType = $existing.properties.authType; Target = $existing.properties.target
    Audience = $existing.properties.audience
} | Format-List

### 7.2 — Register the prompt agent

Update `ALLOWED_TOOLS` in `src/agent/register_agent.py` first if you exposed your own stored
procedures. The three generic tools stay; custom tool names are snake_case derived from the entity
name — entity `GetExampleById` becomes tool `get_example_by_id`.

In [ ]:
$env:AZURE_AI_PROJECT_ENDPOINT      = "https://$foundryAccount.services.ai.azure.com/api/projects/$foundryProject"
$env:AZURE_AI_MODEL_DEPLOYMENT_NAME = $modelDeployment
$env:MCP_ENDPOINT                   = "https://$mcpFqdn/mcp"
$env:MCP_PROJECT_CONNECTION_NAME    = 'sql-mcp'

Write-Host ("Project : {0}" -f $env:AZURE_AI_PROJECT_ENDPOINT)
Write-Host ("MCP     : {0}" -f $env:MCP_ENDPOINT)
Write-Host ''

Push-Location $repoRoot
try { & ./scripts/register-demo-agent.ps1 } finally { Pop-Location }

---
## Phase 8 — Close the public path and validate

### 8.1 — Disable the SQL MI public endpoint

Only once the agent answers. Check nothing else depends on it first — it is often enabled for an
on-premises SSMS server.

In [ ]:
$mi = Get-SqlMi
if (-not $mi.publicDataEndpointEnabled) { Write-Reused 'public endpoint already disabled' }
else {
    Write-Note 'publicDataEndpointEnabled is currently True.'
    Write-Host 'Confirm nothing else uses it, then run:' -ForegroundColor Cyan
    Write-Host ("  az sql mi update -g {0} -n {1} --public-data-endpoint false" -f $rg, $miName) -ForegroundColor Cyan
    # az sql mi update -g $rg -n $miName --public-data-endpoint false -o none
}

### 8.2 — Final validation

In [ ]:
$checks = [System.Collections.Generic.List[object]]::new()
function V($n, $ok, $d) { $checks.Add([pscustomobject]@{ Check = $n; Status = $(if ($ok) { 'PASS' } else { 'FAIL' }); Detail = $d }) }

$acrPublic = az acr show --name $acrName --query publicNetworkAccess -o tsv
V 'Registry private only' ($acrPublic -eq 'Disabled') "publicNetworkAccess=$acrPublic"

$e = Get-AzJson @('containerapp','env','show','-g',$rg,'-n',$envName)
V 'Environment VNet-injected' ($null -ne $e.properties.vnetConfiguration.infrastructureSubnetId) (($e.properties.vnetConfiguration.infrastructureSubnetId -split '/')[-1])

$revs = Get-AzJson @('containerapp','revision','list','-n',$appName,'-g',$rg)
$latest = $revs | Sort-Object { $_.properties.createdTime } -Descending | Select-Object -First 1
V 'Revision running' ($latest.properties.runningState -eq 'Running') $latest.properties.runningState

$a = Get-AzJson @('containerapp','show','-g',$rg,'-n',$appName)
$cs = ($a.properties.template.containers[0].env | Where-Object { $_.name -eq 'DATABASE_CONNECTION_STRING' }).value
V 'VNet-local SQL endpoint' (($cs -notmatch '\.public\.') -and ($cs -match ',1433')) 'no .public. infix, port 1433'

$mi2 = Get-SqlMi
V 'SQL MI public endpoint off' (-not $mi2.publicDataEndpointEnabled) ("enabled={0}" -f $mi2.publicDataEndpointEnabled)

$cfg = Get-Content (Join-Path $repoRoot 'src/mcp-server/dab-config.json') -Raw | ConvertFrom-Json
$demo = @($cfg.entities.PSObject.Properties.Name | Where-Object { $cfg.entities.$_.source.object -match 'vw_transfer_|vw_client_account' })
V 'Config uses your objects' ($demo.Count -eq 0) $(if ($demo.Count) { "still demo: $($demo -join ', ')" } else { 'no demo entities' })

$checks | Format-Table -AutoSize -Wrap

Write-Host ''
Write-Host 'Manual checks that complete the evidence:' -ForegroundColor Cyan
Write-Host '  1. Connect to the .public. FQDN on 3342        -> must FAIL'
Write-Host '  2. Ask the agent a question in the playground   -> must WORK'
Write-Host '  3. Ask the agent for data from a base table     -> must be REFUSED'
Write-Host '  4. Ask the agent to change something            -> must be REFUSED'
Write-Host ''
Write-Host 'Checks 3 and 4 are what a security review will ask for. Capture the output.' -ForegroundColor Cyan

---
## Removing what this created

Safe order. Does not touch SQL MI or, when it was pre-existing, the virtual network.

```powershell
az containerapp delete -n $appName -g $rg --yes
az containerapp env delete -n $envName -g $rg --yes
az network private-endpoint delete -n "pe-$acrName" -g $vnetRg
az acr delete -n $acrName -g $rg --yes
az network vnet subnet delete -g $vnetRg --vnet-name $vnetName -n $acaSubnet
az network vnet subnet delete -g $vnetRg --vnet-name $vnetName -n $peSubnet
# SQL MI, Foundry and the Entra application are deliberately not deleted here.
```

To restore the demo DAB configuration:

```powershell
Copy-Item "$repoRoot/src/mcp-server/dab-config.json.demo-backup" "$repoRoot/src/mcp-server/dab-config.json" -Force
```